# 2. MESA Datagen

## Prerequisites

- AWS credentials
- MESA Runtime credentials

## Overview

Datagen sits alongside finetune as the 'build' components of MESA (below). As with docsynth, a key part of the datagen process is to derive a suitable prompt, in this case one that can elicit a comprehensive standardised version of each synthetic document. Unlike docsynth, prompts aren't derived dynamically but hardcoded to capture domain insight, and thus form versionable assets that are fed into the datagen component. The schema itself is also a key asset, capturing key domain knowledge of the types of data (often biomarkers) likely contained in the free-text being processed.


With these two assets, datagen effectively becomes middleware, interfacing with external services to derive document:schema pairs, and parse, check and reformat them as required, prior to them being used for finetuning.

⭐ Schemas, prompts and docsynth assets -- which capture the domain knowledge of specialists and an understanding of language models for comprehensive standardisation -- are a key contribution of MESA.

<p align="center">
  <img src="https://londonaicentre.github.io/MESA-Build/repo.png" width="700" alt="Flowchart of the document generation pipeline">
</p>

As with docsynth, new prompts and schemas can be developed for new domains. Again, as with docsynth, this library also contains prompts and schemas for existing domains.

> 📓 In keeping with the running example, we will work with an oncology schema and associated prompts in this workbook.

## Steps

1. Import the relevant datagen packages: we want to remind ourselves what our docsynth outputs are called, use batch inference on MESA Runtime again, and then upload our data.

In [ ]:
from datagen import BedrockBatchGenerator, DocumentLoader, MESARuntimeBatchGenerator, TrainingDataUploader

2. Import the relevant asset packages: we want to use 'oncoschema', which includes the prompts and the schema itself. We also want to instantiate the prompt wrapper class. 

In [ ]:
from oncoschema.prompt_builder import PromptBuilder
from oncoschema import Schema
prompt_builder: PromptBuilder = PromptBuilder()

3. Import some utility classes to help us work with the library.

In [ ]:
import os
from pathlib import Path
from uuid import UUID
from dotenv import load_dotenv
load_dotenv()

4. The last step during docsynth uploaded a set of synthetic documents to AWS. We can use a tool from the datagen library to identify what the name of this upload is.

In [ ]:
DocumentLoader().list_available_document_batches()

5. We now want to create a reusable datagen batch generator object. To do so, we supply the prompt to use (system prompt) from our prompt assets, the schema from those same assets, the name of our schema, the name of the document batch we generated from docsynth, our MESA Runtime credentials and the model we want to use. Creating this object will automatically download this batch for us.

In [ ]:
generator: MESARuntimeBatchGenerator = MESARuntimeBatchGenerator(
    prompt_builder.build_datagen_prompt(),
    Schema,
    "oncoschema",
    ["test_batch-2026-07-05-001.tar.gz"],
    os.environ["LLM__MESA_RUNTIME__BASE_URL"],
    os.environ["LLM__MESA_RUNTIME__USERNAME"],
    os.environ["LLM__MESA_RUNTIME__PASSWORD"],
    "qwen3.5",
)

6. Run the following code to start batch generation of document:schema pairs, with batch inference on MESA Runtime, which runs open-weight models on in-house compute and only needs access to our bucket.

>❗Batch inference can be a slow-running process. If you would prefer to skip this step, instead run `cp .job_id.example.json data/datagen/.job_id.json`.

In [ ]:
run_id: UUID = generator.generate_via_batch(100, os.environ["BUCKET"])

7. (Optional) Inspect `data/datagen/anthropic_batch_job.jsonl` to see the prompts we are sending to MESA Runtime to generate our finetuning pairs.

8. Because this is a batch job, we may need to wait before we can process the outputs. We can check the status of the run by running the command below.

In [ ]:
generator.get_batch_run(run_id)

### Alternative: AWS Bedrock

Steps 5 to 8 can instead be run with AWS Bedrock batch processing (Claude Sonnet), which needs a Bedrock execution role (`BEDROCK_EXECUTION_ROLE`) as well as access to our bucket. To do so, run the cells below in place of steps 5 to 8, then continue from step 9.

>❗Bedrock batch inference can be an expensive process.

In [ ]:
generator: BedrockBatchGenerator = BedrockBatchGenerator(
    prompt_builder.build_datagen_prompt(),
    Schema,
    "oncoschema",
    "sonnet4",
    ["test_batch-2026-07-05-001.tar.gz"],
)

In [ ]:
generator.generate_via_batch(100, os.environ["BUCKET"], os.environ["BEDROCK_EXECUTION_ROLE"])

In [ ]:
generator.check_batch_output_status(os.environ["BUCKET"])

9. Once the above check confirms our batch output is ready, we can download it, parse it and output it in a format for the rest of the pipeline. You may notice some validation failures reported; this is understandable when generating large amounts of training data, but you'll also notice that we catch and exclude failing pairs so that they don't pollute our training data.

In [ ]:
generator.extract_batch_output(os.environ["BUCKET"])

10. (Optional) Inspect `data/datagen/samples` to see the format of the training pairs we have generated.

11. Passing much of the same data we have passed previously, we now upload the training data for use by the next component and then move to the [next notebook](../finetune/finetune.ipynb).

In [ ]:
TrainingDataUploader.upload(
    schema=Schema,
    schema_name="oncoschema",
    system_prompt=prompt_builder.build_main_prompt(),
    short_description="oncotest-batch",
    long_description="Training data for OncoTest",
    input_folder=Path("./data/datagen/samples"),
)